# SAC - Pusher & Reacher

In [ ]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo, FlattenObservation, FilterObservation

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

import os
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import random
from tqdm import tqdm

os.environ['KMP_DUPLICATE_LIB_OK']='True'

## Probar ambiente

env = gym.make("Pusher-v5", max_episode_steps=200, render_mode="human") #Reacher-v5
for i in range(10):
    env.reset()
    done=False
    
    while not done:
        _, reward, _, done, _ = env.step(env.action_space.sample())
env.close()

In [ ]:
env.action_space

In [ ]:
env.observation_space

## Bufer de experiencias

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.memory = deque([], capacity)

    def __len__(self):
        return len(self.memory)
    
    def append(self, transition):
        self.memory.append(transition)

    def can_sample(self, batch_size):
        return len(self.memory) > batch_size
    
    def sample(self, batch_size):
        assert self.can_sample(batch_size)
        batch = random.sample(self.memory, batch_size)
        batch = zip(*batch)
        return [torch.cat(item) for item in batch]

## Actualizacion Suave y dura

In [ ]:
def soft_update(target, source, tau):
    for target_param, source_param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(target_param.data * (1.0 - tau) + source_param*tau)

def hard_update(target, source):
    for target_param, source_param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_(source_param.data)

## Creaar redes actor critico

In [ ]:
class Dqn(nn.Module):
    def __init__(self, state_dims, action_dims):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dims + action_dims, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, state, action):
        if isinstance(state, np.ndarray):
            state = torch.from_numpy(state)
        if isinstance(action, np.ndarray):
            action = torch.from_numpy(action)
        vector = torch.hstack([state, action]).to(state.device)
        return self.net(vector.float())
        
class GradientPolicy(nn.Module):
    def __init__(self, state_dims, action_dims, max):
        super().__init__()
        self.register_buffer('max', torch.tensor(max_action))

        self.net = nn.Sequential(
            nn.Linear(state_dims, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU()
        )

        self.mu = nn.Linear(256, action_dims)
        self.std = nn.Linear(256, action_dims)

    def forward(self, state):
        if isinstance(state, np.ndarray):
            state = torch.from_numpy(state).float()

        state = state.to(self.net[0].weight.device)
        
        x = self.net(state)
        mu = self.mu(x)
        std = self.std(x)
        std = F.softplus(std)
        
        dist = torch.distributions.Normal(mu, std)
        action = dist.rsample()
        log_prob = dist.log_prob(action)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        log_prob -= (2* (np.log(2) - action - F.softplus(-2 * action))).sum(dim=-1, keepdim=True)
        
        action = torch.tanh(action) * self.max
        
        return action, log_prob

## Crear algoritmo SAC

In [ ]:
import itertools
class Sac:
    def __init__(self, state_dims, action_dims, max_action, 
                 lr=1e-3, gamma=0.99, tau=0.005, alpha=0.02, 
                 samples_per_epoch=1000, batch_size = 1024):
        self.state_dims = state_dims
        self.action_dims = action_dims
        self.max_action = max_action

        self.qnet1 = Dqn(state_dims, action_dims).to(device)
        self.qnet2 = Dqn(state_dims, action_dims).to(device)
        self.policy = GradientPolicy(state_dims, action_dims, max_action).to(device)

        self.target_qnet1 = Dqn(state_dims, action_dims).to(device)
        self.target_qnet2 = Dqn(state_dims, action_dims).to(device)
        self.target_policy = GradientPolicy(state_dims, action_dims, max_action).to(device)
        
        self.batch_size = batch_size
        self.lr = lr
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha
        self.samples_per_epoch = samples_per_epoch
        self.device = device

        self.qnet1_optimizer = AdamW(self.qnet1.parameters(), lr=self.lr)
        self.qnet2_optimizer = AdamW(self.qnet2.parameters(), lr=self.lr)
        self.policy_optimzier = AdamW(self.policy.parameters(), lr=self.lr)

    def choose_action(self, state):
        if isinstance(state, np.ndarray):
            state = torch.from_numpy(state).unsqueeze(0).float().to(self.device)
        
        self.policy.eval()
        action, log_prob = self.policy(state)
        self.policy.train()
        return action   

    def update_params(self, batch):
        states, actions, rewards, truncs, next_states = batch

        #Dqn
        action_values1 = self.qnet1(states, actions)
        action_values2 = self.qnet2(states, actions)

        target_actions, target_log_probs = self.target_policy(next_states)

        next_action_values = torch.min(
            self.target_qnet1(next_states, target_actions),
            self.target_qnet2(next_states, target_actions)
        )
        next_action_values[truncs] = 0.0

        expected_action_values = rewards + self.gamma * (next_action_values - 
                                                         self.alpha*target_log_probs).detach()

        q_loss1 = F.smooth_l1_loss(action_values1, expected_action_values)
        q_loss2 = F.smooth_l1_loss(action_values2, expected_action_values)

        q_loss = q_loss1 + q_loss2
        self.qnet1_optimizer.zero_grad()
        self.qnet2_optimizer.zero_grad()
        q_loss.backward()
        self.qnet1_optimizer.step()
        self.qnet2_optimizer.step()


        #policy
        actions, log_probs = self.policy(states)
        action_values = torch.min(self.qnet1(states, actions), self.qnet2(states, actions))
        policy_loss = (self.alpha * log_probs - action_values).mean()
        self.policy_optimzier.zero_grad()
        policy_loss.backward()
        self.policy_optimzier.step()
        

        soft_update(self.target_qnet1, self.qnet1, self.tau)
        soft_update(self.target_qnet2, self.qnet2, self.tau)
        soft_update(self.target_policy, self.policy, self.tau)

        return q_loss, policy_loss

## Entrenar en los ambientes Reacher y Pusher

In [ ]:
if torch.cuda.is_available():
    print("Using CUDA device:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available")

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
env_name = "Pusher-v5"

state_dims, action_dims, max_action = env.observation_space.shape[0], env.action_space.shape[0], env.action_space.high
if env_name == "Reacher-v5":
    hyperparams = {
        "lr": 1e-3,
        "gamma": 0.99,
        "tau": 0.005,
        "alpha": 0.02,
        "samples_per_epoch":1000, 
        "batch_size":1024
    }
    max_episode_steps = 50
elif env_name == "Pusher-v5":
    hyperparams = {
        "lr": 3e-4,
        "gamma": 0.99,
        "tau": 0.005,
        "alpha": 0.2,
        "samples_per_epoch":1000, 
        "batch_size": 256
    }
    max_episode_steps = 100
else:
    print("Usar hiperparametros por defecto o probar los suyos")
env = gym.make(env_name, max_episode_steps = max_episode_steps)
agent = Sac(state_dims, action_dims, max_action, **hyperparams)
replay_buffer = ReplayBuffer(1_000_000)
episodes=1000
timesteps=0

return_list = []
for i in range(10):
    with tqdm(total=int(episodes/10), desc = 'Iteracion %d' % i) as pbar:
        for episode in range(int(episodes/10)):
            state, _ = env.reset()
            state = torch.tensor(state).unsqueeze(0).float().to(device)
            episode_return=0
            while True:
                timesteps+=1
                action = agent.choose_action(state).squeeze().detach().cpu().numpy()
                next_state, reward, done, trunc, info = env.step(action)
                episode_return+=reward
                next_state = torch.tensor(next_state).unsqueeze(0).float().to(device)
                reward = torch.tensor(reward).view(1, -1).float().to(device)
                trunc = torch.tensor(trunc).view(1, -1).to(device)
                action = torch.tensor(action).view(1, -1).float().to(device)
                replay_buffer.append([state, action, reward, trunc, next_state])
                state=next_state

                episode_q_loss = 0
                episode_policy_loss = 0
        
                if replay_buffer.can_sample(agent.batch_size):
                    batch = replay_buffer.sample(agent.batch_size)
                    q_loss, policy_loss = agent.update_params(batch)
                    episode_q_loss+=q_loss
                    episode_policy_loss+=policy_loss

                if trunc:
                    break
            return_list.append(episode_return)
            if (episode+1) % 10 == 0:
                pbar.set_postfix({'Episodio': '%d' % (episodes/10 * i + episode + 1),
                                  'Retorno': '%.3f' % np.mean(return_list[-10:]), 
                                  'q_loss': '%.3f' % episode_q_loss, 
                                  'policy_loss': '%.3f' % episode_policy_loss})
            pbar.update(1)
            if (episode + 1) % 100 == 0:
                if env_name == "Pusher-v5":
                    torch.save(agent.policy.state_dict(), f"models\env_pusher\sac_pusher_ep{episode}.pt")
                    print(f"...... Modelo para el episodio {episode} es guardado en pusher ......")
                elif env_name == "Reacher-v5":
                    torch.save(agent.policy.state_dict(), f"models\env_reacher\sac_reacher_ep{episode}.pt")
                    print(f"...... Modelo para el episodio {episode} es guardado en reacher ......")
                else:
                    print("Error - ambiente no considerado. No se guardo el modelo")
                    break

In [ ]:
def moving_average(a, window_size):
    cumulative_sum = np.cumsum(np.insert(a, 0, 0)) 
    middle = (cumulative_sum[window_size:] - cumulative_sum[:-window_size]) / window_size
    r = np.arange(1, window_size-1, 2)
    begin = np.cumsum(a[:window_size-1])[::2] / r
    end = (np.cumsum(a[:-window_size:-1])[::2] / r)[::-1]
    return np.concatenate((begin, middle, end))

In [ ]:
episodes_list = list(range(len(return_list)))
plt.plot(episodes_list, return_list)
plt.xlabel('Episodios')
plt.ylabel('Retornos')
plt.title('SAC en {}'.format(env_name))
plt.show()

mv_return = moving_average(return_list, 9)
plt.plot(episodes_list, mv_return)
plt.xlabel('Episodios')
plt.ylabel('Retornos')
plt.title('SAC en {}'.format(env_name))
plt.show()

## Probar el agente SAC entrenado

In [ ]:
env = gym.make(env_name, max_episode_steps=200, render_mode="human")

model = GradientPolicy(env.observation_space.shape[0], env.action_space.shape[0], env.action_space.high)
if env_name == "Pusher-v5":
    model.load_state_dict(torch.load("models\env_pusher\sac_pusher_ep99.pt", weights_only=True))
elif env_name =="Reacher-v5":
    model.load_state_dict(torch.load("models\env_reacher\sac_reacher_ep99.pt", weights_only=True))


for i in range(30):
    state, _ = env.reset()    
    while True:
        action, _ = model(state)
        next_state, reward, done, trunc, info = env.step(action.detach().numpy())
        state=next_state
        if trunc:
            break
env.close()